# WaterTracker : entraînement du U-Net de détection de l'eau (Colab)

Avant de lancer : **Exécution → Modifier le type d'exécution → GPU T4**, et déposez `watertracker_unet.zip` à la racine de votre Google Drive (« Mon Drive »).

Puis **Exécution → Tout exécuter**. Durée prévue : 15 à 30 minutes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, torch
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN (activer le GPU T4)')

In [ ]:
# Décompression des imagettes et du code (environ 1 minute)
!rm -rf /content/wt && mkdir -p /content/wt
!unzip -q /content/drive/MyDrive/watertracker_unet.zip -d /content/wt
!ls /content/wt/data/unet/chips | wc -l

In [ ]:
%cd /content/wt
# Entraînement : 40 passes, la meilleure en validation est gardée
!python -W ignore -m ml.unet_model --data data/unet --out ml/runs/unet_latest --epochs 40 --batch 32

In [ ]:
# Évaluation sur les blocs de test, jamais vus pendant l'entraînement
!python -W ignore -m ml.unet_eval --data data/unet --model ml/runs/unet_latest/model.pt --split test

In [ ]:
# Copie des résultats sur le Drive : à télécharger ensuite dans backend/ml/runs/unet_latest/
!mkdir -p /content/drive/MyDrive/watertracker_unet_results
!cp ml/runs/unet_latest/* /content/drive/MyDrive/watertracker_unet_results/
!ls -la /content/drive/MyDrive/watertracker_unet_results